# Systematic uncertainty summary (all sources)

Load Product **B** covariance products from the individual source notebooks /
chunked drivers, export `CategorySummary/category_syst_summary.npz`, and plot
**total** fractional uncertainties + cov/corr matrices in the style of
`systematics-genie-inspect.ipynb` Section 2 — with one curve per **source**
(Flux, G4, MCstat, Detector, Cosmics, GENIE, Exposure, Targets).

**Flat normalization scales** (when `INCLUDE_FLAT` is true; same % in every bin,
**100% correlated across bins** because each is one global multiplier):
- **Exposure** (POT): **2%**
- **Targets** (number of targets): **1%**

Exposure and Targets are **independent** of each other (separate categories).

**Detector** is a single source: the combined NPZ from
`systematics-detector.ipynb` (`Detector/detector_syst_dict.npz` =
WireMod YZ + XTXW + DENT). Do not point this notebook at WireMod/DENT roots
separately.

Two sets: **rate** and **xsec**. Only the GENIE component changes between them
(`genie_rate` vs `genie_xsec`).

Per-source digs: `systematics-flux.ipynb`, `systematics-g4.ipynb`,
`systematics-mcstat.ipynb`, `systematics-genie-inspect.ipynb`,
`systematics-cosmic.ipynb`, `systematics-detector.ipynb`, …

Helpers: `syst_summary_inspect.py`, `syst_category_summary.py`.


In [ ]:
%load_ext autoreload
%autoreload 2


In [ ]:
import os
import shutil
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np

REPO = Path("/exp/sbnd/app/users/munjung/xsec/freeze/cafpyana")
sys.path.insert(0, str(REPO))

from analysis_village.numucc_1p0pi.dataset_locations import PLOTS_BASE, default_syst_disk_root
from analysis_village.numucc_1p0pi.final_selected_evt_vars import (
    CORE_SELECTED_EVT_VARIABLE_CONFIGS,
    with_final_selected_evt_variables,
)
from analysis_village.numucc_1p0pi.syst_category_summary import (
    CAT_COSMICS,
    CAT_FLUX,
    CAT_GENIE_RATE,
    CAT_GENIE_XSEC,
    CAT_NTARGETS,
    CAT_POT,
    NTARGETS_FRAC_UNC_PCT,
    POT_FRAC_UNC_PCT,
    TOTAL_RATE,
    TOTAL_XSEC,
    category_summary_npz_path,
    export_category_syst_summary,
    load_category_syst_summary,
)
from analysis_village.numucc_1p0pi.syst_disk_layout import normalized_root
from analysis_village.numucc_1p0pi.syst_multisim_common import build_var_configs
from analysis_village.numucc_1p0pi.syst_summary_inspect import (
    load_source_payloads,
    plot_total_source_section,
)
from analysis_village.numucc_1p0pi.utils import dpi
from analysis_village.numucc_1p0pi.variable_configs import VariableConfig

plt.style.use("presentation.mplstyle")


## Configuration

Set a **unified** `SYST_DISK_ROOT` and/or per-source roots. Explicit
`SOURCE_DISK_ROOTS` entries win when present.

For **Detector**, point at the tree that contains
`Detector/detector_syst_dict.npz` from `systematics-detector.ipynb`
(typically the same `SYST_DISK_ROOT` / PRL `productB_sel_mup` used there).


In [ ]:
# --- disk roots ---
# Unified tree (optional fallback for any source not listed below)
SYST_DISK_ROOT = os.environ.get(
    "NUMUCC_SYST_DISK_ROOT",
    "/exp/sbnd/data/users/munjung/xsec/numucc_1p0pi/PRL/systematics/productB_sel_mup",
)
if not SYST_DISK_ROOT:
    SYST_DISK_ROOT = str(default_syst_disk_root())
SYST_DISK_ROOT = str(Path(SYST_DISK_ROOT).expanduser())

# Per-source notebook / campaign roots (override unified when set and present)
_PLOTS = Path(PLOTS_BASE)
SOURCE_DISK_ROOTS = {
    # "Flux": _PLOTS / "systematics-notebook-flux-YYYYMMDD",
    # "G4": _PLOTS / "systematics-notebook-g4-YYYYMMDD",
    # "MCstat": _PLOTS / "systematics-notebook-mcstat-YYYYMMDD",
    # "GENIE": Path("/exp/sbnd/data/users/munjung/xsec/numucc_1p0pi/syst_disk_…"),
    # "Cosmics": _PLOTS / "systematics-notebook-cosmic-YYYYMMDD",
    # Combined Detector (WireMod+DENT) from systematics-detector.ipynb:
    # "Detector": _PLOTS / "systematics-final",
}
# Env overrides, e.g. SYST_FLUX_ROOT=/path/to/systematics-notebook-flux-…
for _src, _env in (
    ("Flux", "SYST_FLUX_ROOT"),
    ("G4", "SYST_G4_ROOT"),
    ("MCstat", "SYST_MCSTAT_ROOT"),
    ("GENIE", "SYST_GENIE_ROOT"),
    ("Cosmics", "SYST_COSMICS_ROOT"),
    ("Detector", "SYST_DETECTOR_ROOT"),
):
    _v = os.environ.get(_env)
    if _v:
        SOURCE_DISK_ROOTS[_src] = Path(_v)

VAR_SET = os.environ.get("SUMMARY_VAR_SET", "final")
# Or set explicitly, e.g. [VariableConfig.muon_end_x(), VariableConfig.tki_del_Tp()]
var_configs = build_var_configs(VAR_SET)

COV_TYPES = ["rate", "xsec"]
# Flat normalization scales (fully correlated across bins): POT 2%, Targets 1%.
INCLUDE_FLAT = True
EXPOSURE_FLAT_PCT = float(POT_FRAC_UNC_PCT)    # 2.0
TARGETS_FLAT_PCT = float(NTARGETS_FRAC_UNC_PCT)  # 1.0
assert EXPOSURE_FLAT_PCT == 2.0 and TARGETS_FLAT_PCT == 1.0
SAVE_FIGS = True
FIG_DPI = int(os.environ.get("SUMMARY_FIG_DPI", str(dpi if dpi < 200 else 140)))
PLOT_SOURCE_HEATMAPS = True  # frac cov/corr per source (like GENIE modes in inspect §2)
PLOT_TOTAL_HEATMAPS = True

OUT_DIR = Path(PLOTS_BASE) / "syst_uncertainty_summary"
OUT_DIR.mkdir(parents=True, exist_ok=True)

print("SYST_DISK_ROOT =", SYST_DISK_ROOT)
print("SOURCE_DISK_ROOTS =", {k: str(v) for k, v in SOURCE_DISK_ROOTS.items()})
print("OUT_DIR =", OUT_DIR)
print(f"{len(var_configs)} variables ({VAR_SET}):", [vc.var_save_name for vc in var_configs])
print(
    f"INCLUDE_FLAT={INCLUDE_FLAT}: Exposure={EXPOSURE_FLAT_PCT:.1f}%, "
    f"Targets={TARGETS_FLAT_PCT:.1f}% (each fully correlated across bins)"
)


## Load source covariances

`Detector` loads `Detector/detector_syst_dict.npz` from
`systematics-detector.ipynb` (one combined source).


In [ ]:
payloads = load_source_payloads(
    unified_root=SYST_DISK_ROOT,
    source_roots=SOURCE_DISK_ROOTS,
)
print("roots used:", payloads.get("roots_used"))
if payloads.get("detector_npz") is not None:
    _dk = sorted(k for k in dict(payloads["detector_npz"]) if str(k).startswith("detector"))
    print("Detector keys:", _dk)
else:
    print("Detector: not loaded — run systematics-detector.ipynb and set Detector root")


## Export category summary NPZ

Writes `CategorySummary/category_syst_summary.npz` under `SYST_DISK_ROOT` (and a
copy under `OUT_DIR`) for downstream overlays / unfolding
(`load_category_syst_summary`, `get_category_summary_syst_unc`).


In [ ]:
CATEGORY_SUMMARY_NPZ = category_summary_npz_path(SYST_DISK_ROOT)
CATEGORY_SUMMARY_OUT = OUT_DIR / "category_syst_summary.npz"

_nominal_mc_by_var = {}
for vc in var_configs:
    cache = OUT_DIR / f"summary_mc_cv__{vc.var_save_name}.npy"
    if cache.is_file():
        _nominal_mc_by_var[vc.var_save_name] = np.load(cache)
if _nominal_mc_by_var:
    print("Export includes absolute cov for:", sorted(_nominal_mc_by_var))
else:
    print("Export: cov_frac + corr only (optional summary_mc_cv__*.npy not found)")

export_manifest = export_category_syst_summary(
    str(CATEGORY_SUMMARY_NPZ),
    var_configs,
    flux_npz=payloads["flux_npz"],
    g4_npz=payloads["g4_npz"],
    cosmics_npz=payloads["cosmics_npz"],
    detector_npz=payloads["detector_npz"],
    mcstat_npz=payloads["mcstat_npz"],
    genie_blob=payloads["genie_blob"],
    syst_disk_root=SYST_DISK_ROOT,
    include_flat=INCLUDE_FLAT,
    nominal_mc_by_var=_nominal_mc_by_var,
)
shutil.copy2(CATEGORY_SUMMARY_NPZ, CATEGORY_SUMMARY_OUT)
print("Wrote", CATEGORY_SUMMARY_NPZ)
print("Copied", CATEGORY_SUMMARY_OUT)
print("Variables:", export_manifest["variables"])
if export_manifest.get("skipped"):
    print("Skipped:", export_manifest["skipped"])

# Quick sanity check on first exported variable that has flux
_summary = load_category_syst_summary(str(CATEGORY_SUMMARY_NPZ))
_vsns = export_manifest["variables"]
if _vsns:
    _vsn = _vsns[0]
    _v = _summary["by_var"][_vsn]
    print(f"Check {_vsn}: categories = {sorted(_v['categories'])}")
    if TOTAL_RATE in _v:
        print(f"  {TOTAL_RATE} mean % = {_v[TOTAL_RATE]['frac_unc_pct'].mean():.3f}")
    if TOTAL_XSEC in _v:
        print(f"  {TOTAL_XSEC} mean % = {_v[TOTAL_XSEC]['frac_unc_pct'].mean():.3f}")
    if INCLUDE_FLAT:
        for _key, _lab, _exp in (
            (CAT_POT, "Exposure", EXPOSURE_FLAT_PCT),
            (CAT_NTARGETS, "Targets", TARGETS_FLAT_PCT),
        ):
            if _key not in _v["categories"]:
                print(f"  WARNING: missing flat category {_lab} ({_key})")
                continue
            _pct = float(np.asarray(_v["categories"][_key]["frac_unc_pct"]).ravel()[0])
            _cov = np.asarray(_v["categories"][_key]["cov_frac"])
            _vflat = (_exp / 100.0) ** 2
            _scale = bool(np.allclose(_cov, _vflat))
            _corr = np.asarray(_v["categories"][_key]["corr"])
            _ones = np.ones_like(_corr) if _corr.size else _corr
            _corr_ok = _corr.size <= 1 or bool(np.allclose(_corr, _ones))
            print(
                f"  {_lab} ({_key}): {_pct:.3f}% "
                f"(expect {_exp:.1f}%, scale-cov constant={_scale}, corr=1={_corr_ok})"
            )


## Total uncertainty by source (rate + xsec)

For each analysis variable:

1. Fractional uncertainty curves for every available source + **Total**
2. Fractional covariance + correlation heatmaps for **Total** (and each source
   when `PLOT_SOURCE_HEATMAPS` is on)

Rate and xsec figures are identical except for the GENIE curve.


In [6]:
plot_total_source_section(
    payloads,
    var_configs,
    kinds=COV_TYPES,
    out_dir=OUT_DIR,
    save_figs=SAVE_FIGS,
    dpi=FIG_DPI,
    include_flat=INCLUDE_FLAT,
    plot_source_heatmaps=PLOT_SOURCE_HEATMAPS,
    plot_total_heatmaps=PLOT_TOTAL_HEATMAPS,
)
